In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score, recall_score,f1_score
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
%matplotlib inline

In [ ]:
#Reads and previews the initial 10 rows of a 'Lung Cancer.csv' dataset using Pandas
df=pd.read_csv('Lung Cancer.csv')
df.head(10)

In [ ]:
#Data pre processing: It involves cleaning, transforming, and organizing raw data to prepare it for analysis or model 

In [ ]:
df.shape

In [ ]:
df=df.dropna()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
#A boxplot displays the distribution of a numerical variable, showing its median, quartiles, and potential outliers, 
#offering insights into the data's spread and central tendency
sns.boxplot(df['AGE'])

In [ ]:
#Removing outliers
#Identifies and extracts outliers in the 'AGE' column using the interquartile range (IQR) method
df_age=df['AGE']
q3=df_age.quantile(0.75)
q1=df_age.quantile(0.25)
iqr=q3-q1
ll=q1-(1.5*iqr)
ul=q3+(1.5*iqr)
age_outliers = df_age[(df_age<ll) | (df_age>ul)]

In [ ]:
(age_outliers)

In [ ]:
outlier=[22,238,261,277]

In [ ]:
df.drop(index=outlier, inplace= True)

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
#Label encoding transforms categorical data into numerical format by assigning a unique numeric label to each category
le=LabelEncoder()
df['LUNG_CANCER']=le.fit_transform(df['LUNG_CANCER'])
df['GENDER']=le.fit_transform(df['GENDER'])

In [ ]:
df['LUNG_CANCER'].head(10)

In [ ]:
df['LUNG_CANCER'].value_counts()

In [ ]:
sns.countplot(df['LUNG_CANCER'])#Plots counts of categorical values

In [ ]:
df1=df.drop('AGE',axis=1)

In [ ]:
dict1={}

In [ ]:
#Calculates chi-square statistics for each column in df1 related to 'LUNG_CANCER' and prints the results, storing 
#chi-square values in dict1.

#Chi-square test assesses independence between categorical variables by comparing observed and expected frequencies in 
#a contingency table.
for column in df1.columns:
    table=pd.crosstab(df[column],df['LUNG_CANCER'])
    chi2,_, _, _ = chi2_contingency(table)
    print(f"Chi-square statistic for {column}: {chi2}")
    dict1[column]=chi2

In [ ]:
#Creates age groups categorization.
#Bins are predefined intervals in data
bins = [44, 60, 81] 
labels = ['44-60', '61-81'] # Label assigned to each bin
df['AGE_group'] = pd.cut(df['AGE'], bins=bins, labels=labels)

In [ ]:
#Prints chi-square summary for age groups' association with lung cancer, storing chi-square value in `dict1['AGE_group']`.
#A contingency table summarizes the distribution of categorical variables, showcasing their joint occurrences.
contingency_table = pd.crosstab(df['AGE_group'], df['LUNG_CANCER'])
print(contingency_table)
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print(chi2)
dict1['AGE_group']=chi2

In [ ]:
#Sorts dictionary by values descendingly
sorted_dict=sorted(dict1.items(),key=lambda item: item[1],reverse=True)
sorted_dict

In [ ]:
df.head()

In [ ]:
x=df.drop(['SMOKING','GENDER','SHORTNESS OF BREATH','AGE','CHRONIC DISEASE','ANXIETY','AGE_group','LUNG_CANCER','YELLOW_FINGERS','FATIGUE '],axis=1)

In [ ]:
y=df['LUNG_CANCER']

In [ ]:
x

In [ ]:
y

In [ ]:
#Balances classes using SMOTE technique, displaying new class distribution.
#SMOTE creates synthetic samples to balance class distribution in machine learning datasets
smote=SMOTE(random_state=42)
x,y=smote.fit_resample(x,y)
y.value_counts()

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
#Logistic Regression:It predicts binary outcomes by modeling the relationship between independent 
#variables and the probability of a particular event

In [ ]:
sns.countplot(y)

In [ ]:
model=LogisticRegression()

In [ ]:
model.fit(x_train,y_train)

In [ ]:
#Checking for overfitting
train_accuracy = model.score(x_train, y_train)
val_accuracy = model.score(x_test, y_test)
print(train_accuracy)
print(val_accuracy)
#No overfitting since these values are close

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
#Calculates the accuracy of predictions on the test set by comparing them to the actual labels.
#Accuracy Score Formula= {Number of Correct Predictions}\{Total Number of Predictions}
accuracy=accuracy_score(y_test,y_pred)

In [ ]:
print(accuracy)

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})


In [ ]:
#Confusion matrix is a table showing the performance of a classification model, summarizing true positive, true negative, 
#false positive, and false negative values
cm=confusion_matrix(y_test,y_pred)

In [ ]:
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

In [ ]:
print(metrics_table)

In [ ]:
#K nearest neighbors:It is a supervised machine learning algorithm that classifies a data point based on the majority 
#class of its k-nearest neighbors in the feature space.
#The Euclidean distance formula is commonly used to measure the distance between data points in K Nearest Neighbors. 
#For two points (x_1, y_1) and (x_2, y_2), the Euclidean distance ((d)) is calculated as

In [ ]:
# Fits a K Nearest Neighbors classifier with \(k=15\) on training data, predicts on the test data, and prints the accuracy 
#score.
knn=KNeighborsClassifier(n_neighbors=15)
knn.fit(x_train,y_train)
y_pred=knn.predict(x_test)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})

In [ ]:
cm=confusion_matrix(y_test,y_pred)

In [ ]:
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

In [ ]:
print(metrics_table)

In [ ]:
#Naive Baye's: Naive Bayes is a probabilistic classification algorithm based on Bayes' theorem, assuming independence 
#among features

In [ ]:
#Fits a Gaussian Naive Bayes model on training data, predicts on the test data, and prints the accuracy score
model=GaussianNB()
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})

In [ ]:
cm=confusion_matrix(y_test,y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

In [ ]:
print(metrics_table)

In [ ]:
#SVM:Support Vector Machine (SVM) is a machine learning algorithm that finds the optimal hyperplane to separate data 
#into classes, maximizing the margin between them

In [ ]:
model = SVC(kernel='rbf', C=1.0, gamma='scale')
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})

In [ ]:
cm=confusion_matrix(y_test,y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

In [ ]:
print(metrics_table)

In [ ]:
#Decision tree: A decision tree is a visual representation of choices and their possible outcomes, used for decision-making
#by following branches of criteria

In [ ]:
#A decision tree classifier is created, trained on the provided training data, used to predict outcomes for test data, and 
# its accuracy is calculated and printed
model=DecisionTreeClassifier(random_state=42)
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})

In [ ]:
cm=confusion_matrix(y_test,y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

In [ ]:
print(metrics_table)

In [ ]:
''' Summary:

1. Data Cleaning and Exploration:
   - The dataset is loaded using Pandas and the initial 10 rows are displayed.
   - Data is cleaned by removing missing values and duplicates.
   - Outliers in the 'AGE' column are identified and removed.

2. Data Preprocessing:
   - Label encoding is applied to 'GENDER' and 'LUNG_CANCER' columns.
   - Age groups are created based on specified bins.

3. Statistical Analysis:
   - Chi-square tests are performed to assess the relationship between categorical variables and the target variable 
      ('LUNG_CANCER').
   - The statistical results are stored in a dictionary and sorted in descending order.

4. Feature Engineering:
   - The dataset is further modified by creating age groups and dropping irrelevant columns.

5. Handling Imbalanced Data:
   - Synthetic Minority Over-sampling Technique (SMOTE) is used to balance the target variable classes.

6. Train-Test Split:
   - The data is split into training and testing sets.

7. Model Training:
   - Logistic Regression, K-Nearest Neighbors (KNN), Gaussian Naive Bayes, Support Vector Machine (SVM), and Decision 
   Tree classifiers are trained on the data.

8. Model Evaluation:
   - The accuracy of each model is evaluated on the test set.
   - Confusion matrices are printed and visualized using seaborn's heatmap.

9. Conclusion/Inference:
   - The conclusion and inference depend on the performance metrics and domain knowledge.
   - The accuracy of each model on the test set is printed, and confusion matrices provide insights into model performance.

10. Parameters Used:
    - Logistic Regression, KNN, Gaussian Naive Bayes, SVM, and Decision Tree models are instantiated with default parameters.
    - For SVM, the 'rbf' kernel is used with C=1.0 and gamma='scale' parameters.
'''
